# AgroSmart — treino do modelo de visão

Treina a rede que estima **quantos por cento da lavoura foram prejudicados** pela estiagem, a partir de imagens do talhão (HU11).

TCC — Engenharia de Computação, Universidade São Judas Tadeu, 2026.

---

## Antes de começar: ligue a GPU

**Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: GPU (T4)**

Sem GPU o treino leva horas; com a T4 gratuita, cerca de 20 minutos.

## O que este notebook faz

1. baixa o código do projeto e a base de imagens;
2. mede o **estimador clássico** (heurística de cor), para ter com o que comparar;
3. treina a rede;
4. gera a tabela e as figuras que entram no documento do TCC;
5. baixa os pesos para você colocar no `.env` do módulo de visão.

Leia [docs/VISAO.md](https://github.com/USJT2026TCC/SmartAgro/blob/main/docs/VISAO.md) para entender **como o percentual é calculado** — as três decisões dessa conta mudam quanto a apólice paga.

## 1. Confirmar a GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("SEM GPU.")
    print("Ambiente de execucao -> Alterar o tipo de ambiente de execucao -> GPU (T4).")
    print("Da para continuar sem GPU, mas o treino vai levar horas.")

## 2. Baixar o código do projeto

In [ ]:
import os
import sys
from pathlib import Path

# Baixa o codigo. Se ja foi baixado, so atualiza: apagar e clonar de novo
# apagaria tambem a base de imagens ja baixada em visao/dados/.
if Path("/content/SmartAgro/.git").exists():
    !git -C /content/SmartAgro pull --ff-only
else:
    !git clone --depth 1 https://github.com/USJT2026TCC/SmartAgro.git /content/SmartAgro

# Todas as celulas seguintes contam com a pasta visao/ como pasta de trabalho.
%cd /content/SmartAgro/visao
!pip install -q -e .

# O Colab so enxerga um pacote instalado com "pip install -e" depois de
# reiniciar o kernel. Em vez de pedir para reiniciar, o codigo entra direto no
# caminho de importacao: sys.path para estas celulas, PYTHONPATH para os
# comandos com "!".
SRC = "/content/SmartAgro/visao/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC

# Se uma tentativa anterior de importar falhou, o Python guardou em cache um
# pacote "visao" vazio: a PASTA visao/ do repositorio, que nao e o pacote.
# Sem limpar, o erro continua mesmo com o caminho corrigido.
for nome in [m for m in sys.modules if m == "visao" or m.startswith("visao.")]:
    del sys.modules[nome]

import visao.indice as indice

print("pacote carregado de:", indice.__file__)
print("classes:", indice.CLASSES)
print("pesos da gravidade:", indice.PESOS_PADRAO)


## 3. Baixar a base de imagens

**Fonte:** SUİÇMEZ, Ç.; YILMAZ, C.; KAHRAMAN, H. T. *UAV-Based Multispectral Maize Dataset for Water Stress 2025 and Common Rust 2025 Detection: Full Dataset with Source Orthomosaics*. Zenodo, **v2.1**, 2026. DOI [10.5281/zenodo.22062459](https://doi.org/10.5281/zenodo.22062459)

**Licença:** CC BY 4.0 — uso livre, exigindo atribuição. Cite a fonte no TCC.

Arquivo: `02_processed_patches.zip`, 545 MB. Usamos só o voo do estresse hídrico: 344 recortes de 224×224, com máscara.

> **Por que a v2.1, e não o subconjunto v1.0 da primeira rodada:** as máscaras são geradas automaticamente por índices de vegetação que usam infravermelho. Na v1.0, esses índices foram calculados com as bandas trocadas, e 56% da lavoura saía como estresse; na v2.1, corrigida pelos autores, são 5,8%. O primeiro treino aprendeu o rótulo errado.

A célula baixa com novas tentativas e **confere o MD5**: a conexão com o Zenodo cai com frequência, e um arquivo truncado só falharia mais adiante.


In [ ]:
import hashlib
import time
from pathlib import Path

import requests

VISAO = Path("/content/SmartAgro/visao")
URL = "https://zenodo.org/api/records/22062459/files/02_processed_patches.zip/content"
MD5_ESPERADO = "851d8b47bc2e941746db9602c72d1cb7"
TAMANHO_ESPERADO = 571_284_472

destino = VISAO / "dados" / "v2.1" / "02_processed_patches.zip"
destino.parent.mkdir(parents=True, exist_ok=True)
parcial = destino.with_suffix(".parcial")


def md5_de(caminho):
    resumo = hashlib.md5()
    with caminho.open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1 << 20), b""):
            resumo.update(bloco)
    return resumo.hexdigest()


if destino.exists() and md5_de(destino) == MD5_ESPERADO:
    print("Base ja baixada e integra.")
else:
    for tentativa in range(1, 8):
        recebido = parcial.stat().st_size if parcial.exists() else 0
        cabecalhos = {"Range": f"bytes={recebido}-"} if recebido else {}
        try:
            with requests.get(URL, headers=cabecalhos, stream=True, timeout=60) as resposta:
                # Retoma so se o servidor aceitar (206). Se responder 200, recomeca
                # do zero: emendar um arquivo novo no fim de um parcial o corrompe.
                if recebido and resposta.status_code != 206:
                    recebido = 0
                resposta.raise_for_status()
                with parcial.open("ab" if recebido else "wb") as saida:
                    for bloco in resposta.iter_content(chunk_size=1 << 20):
                        saida.write(bloco)
        except requests.RequestException as erro:
            print(f"tentativa {tentativa}: a conexao caiu ({erro.__class__.__name__}), retomando")
            time.sleep(5)
            continue

        tamanho = parcial.stat().st_size
        print(f"tentativa {tentativa}: {tamanho:,} de {TAMANHO_ESPERADO:,} bytes")
        if tamanho < TAMANHO_ESPERADO:
            continue
        if md5_de(parcial) == MD5_ESPERADO:
            parcial.replace(destino)
            break
        print("MD5 nao confere; recomecando do zero")
        parcial.unlink()

    if not destino.exists():
        raise RuntimeError("Nao foi possivel baixar a base integra. Rode esta celula de novo.")

print(f"arquivo: {destino}")
print(f"md5....: {md5_de(destino)} (confere)")
print("Fonte: Zenodo, v2.1, DOI 10.5281/zenodo.22062459, CC BY 4.0.")


## 4. Preparar os dados

Extrai **só o voo do estresse hídrico** — doença está fora do escopo do trabalho, e o voo da ferrugem ensinou o primeiro modelo a reconhecer o voo em vez da lesão.

O RGB é montado **pelas bandas nomeadas** do arquivo multiespectral (vermelho = 2, verde = 1, azul = 0). O `.jpg` da base grava essas bandas com vermelho e azul trocados, e não é usado.

A divisão é **espacial, por faixas verticais alternadas** de 512 px: uma faixa a cada quatro vai para a validação. O estresse não se espalha por igual no talhão, e cortar o talhão em dois poria na validação uma região só.


In [ ]:
%cd /content/SmartAgro/visao
!python treino/preparar_dados.py

from pathlib import Path

# Um comando com "!" que falha NAO interrompe o notebook: sem esta conferencia,
# o erro so apareceria na secao seguinte, apontando para o lugar errado.
indice_csv = Path("/content/SmartAgro/visao/dados/preparado/indice.csv")

if not indice_csv.exists():
    raise RuntimeError(
        "A preparacao nao gerou dados/preparado/indice.csv. Leia a mensagem logo acima. "
        "O mais comum e o ZIP nao estar em visao/dados/: rode a secao 3 de novo."
    )

print(f"\nPronto: {indice_csv}")


## 5. Medir o estimador clássico (a linha de base)

Antes de treinar, o número com o qual comparar. A heurística de cor separa solo de planta pelo excesso de verde e planta sadia de planta estressada pelo matiz — sem treino nenhum.

**Guarde esta saída para o TCC.** Ela é o que justifica o modelo treinado existir.

In [ ]:
%cd /content/SmartAgro/visao
!python treino/avaliar_baseline.py


## 6. Treinar

Cerca de 20 minutos em GPU T4, com 30 épocas.

Os pesos são gravados **na melhor época**, e o critério de "melhor" é o **erro do índice de dano**, não a IoU. IoU mede acerto pixel a pixel; o índice é o número que paga. Um modelo pode errar a borda de cada mancha e ainda acertar a proporção de lavoura afetada — e é a proporção que vira dinheiro.

Com uma ressalva que o próprio script aplica: **só qualifica a época que de fato previu as duas classes de estresse.** Um modelo que nunca marca estresse reporta dano zero em tudo e, num conjunto onde a maioria dos recortes tem pouco dano, ganharia um erro médio baixo por acidente — seria uma apólice que nunca paga, com boa métrica.

Se a sessão do Colab cair no meio, basta rodar esta célula de novo: os dados já estão preparados.


In [ ]:
%cd /content/SmartAgro/visao
!python treino/treinar.py --epocas 30 --lote 16 --versao visao-unet-1.0.0

from pathlib import Path

if not Path("/content/SmartAgro/visao/pesos/unet.pt").exists():
    raise RuntimeError("O treino nao gravou pesos/unet.pt. Leia a mensagem logo acima.")


## 7. A tabela do TCC

Compara a linha de base com o modelo treinado **na mesma prova**: os recortes da validação com pelo menos 10% de lavoura, medidos pelo mesmo código (`treino/avaliacao.py`).

> **Leia junto a referência trivial.** Com os rótulos corrigidos, o dano médio da lavoura é de uns 5%. Responder **sempre 0%** já erra só isso. Um estimador que não fique bem abaixo desse número não está enxergando o estresse — está só acompanhando a média. A heurística de cor, por exemplo, erra **mais** do que a resposta trivial.


In [ ]:
%cd /content/SmartAgro/visao
!python treino/avaliar_baseline.py
print()
!python treino/avaliar_modelo.py --pesos pesos/unet.pt


In [ ]:
%cd /content/SmartAgro/visao
import json
from pathlib import Path

historico = json.loads(Path("pesos/unet.historico.json").read_text(encoding="utf-8"))
chave = "erro_do_indice_hidrico"
melhor = min(historico["epocas"], key=lambda e: e[chave])

print(f"Recortes de treino....: {historico['recortes_de_treino']}")
print(f"Recortes de validacao.: {historico['recortes_de_validacao']}")
print(f"Criterio da melhor....: {historico['criterio_da_melhor_epoca']}")
print(f"Melhor epoca..........: {melhor['epoca']} de {len(historico['epocas'])}")
print()
print(f"Erro do indice na melhor epoca: {melhor[chave]:.1%}")
print(f"Responder sempre 0% erraria...: {historico['erro_de_responder_sempre_zero']:.1%}")
print()
print(f"IoU media: {melhor['iou_media']:.3f}")
for classe, valor in melhor["iou_por_classe"].items():
    print(f"  {classe:18} {valor:.3f}")
print()
print("A epoca foi escolhida olhando a propria validacao, entao o erro dela e um")
print("pouco otimista. Para o TCC, vale a comparacao da celula anterior.")


In [ ]:
%cd /content/SmartAgro/visao
import matplotlib.pyplot as plt

epocas = [e["epoca"] for e in historico["epocas"]]
erro_indice = [e["erro_do_indice_hidrico"] for e in historico["epocas"]]
iou = [e["iou_media"] for e in historico["epocas"]]
trivial = historico["erro_de_responder_sempre_zero"]

figura, eixo = plt.subplots(figsize=(8, 4.5))
eixo.plot(epocas, erro_indice, label="erro do indice de dano", linewidth=2)
eixo.plot(epocas, iou, label="IoU media", linewidth=2, linestyle="--")
eixo.axhline(trivial, color="gray", linestyle=":", label="responder sempre 0%")
eixo.set_xlabel("epoca")
eixo.set_ylabel("valor")
eixo.set_title("Treino do segmentador de estresse hidrico (base v2.1)")
eixo.grid(alpha=0.3)
eixo.legend()

figura.tight_layout()
figura.savefig("pesos/curva-de-treino.png", dpi=150)
plt.show()

print("Figura salva em pesos/curva-de-treino.png")


## 8. A figura qualitativa

Imagem, máscara de referência e predição, lado a lado. É o que mostra à banca o que o modelo faz — e também onde ele erra.

In [ ]:
%cd /content/SmartAgro/visao
import csv

import numpy as np
import torch
from matplotlib.colors import ListedColormap
from PIL import Image

from visao.indice import CLASSES, indice_de_dano
from visao.rede import carregar, padronizar

CORES = ListedColormap(["#8d6e63", "#2e7d32", "#fbc02d", "#c62828"])

modelo, versao = carregar("pesos/unet.pt")
raiz = Path("dados/preparado")

with (raiz / "indice.csv").open(encoding="utf-8") as arquivo:
    linhas = [
        l for l in csv.DictReader(arquivo)
        if l["divisao"] == "validacao" and float(l["lavoura"]) >= 0.10
    ]

# Quatro recortes: os de maior dano de referencia e um sem dano, para mostrar
# tanto a deteccao quanto o que o modelo faz com lavoura sadia.
linhas.sort(key=lambda l: float(l["dano"]), reverse=True)
amostras = linhas[:3] + [linhas[-1]]

figura, eixos = plt.subplots(len(amostras), 3, figsize=(9, 3 * len(amostras)))

for linha_do_grafico, registro in zip(eixos, amostras):
    imagem = Image.open(raiz / "images" / registro["imagem"]).convert("RGB")
    verdade = np.asarray(Image.open(raiz / "masks" / registro["mascara"]), dtype=np.int64)

    entrada = torch.from_numpy(np.asarray(imagem, dtype=np.float32) / 255.0).permute(2, 0, 1)
    with torch.no_grad():
        previsto = modelo(padronizar(entrada)[None]).argmax(dim=1)[0].numpy()

    contar = lambda m: {nome: int((m == c).sum()) for c, nome in enumerate(CLASSES)}  # noqa: E731

    linha_do_grafico[0].imshow(imagem)
    linha_do_grafico[0].set_title("imagem", fontsize=10)
    linha_do_grafico[1].imshow(verdade, cmap=CORES, vmin=0, vmax=3)
    linha_do_grafico[1].set_title(f"referencia: dano {indice_de_dano(contar(verdade)):.0%}", fontsize=10)
    linha_do_grafico[2].imshow(previsto, cmap=CORES, vmin=0, vmax=3)
    linha_do_grafico[2].set_title(f"modelo: dano {indice_de_dano(contar(previsto)):.0%}", fontsize=10)

    for celula in linha_do_grafico:
        celula.axis("off")

figura.suptitle("solo (marrom) · saudavel (verde) · estresse leve (amarelo) · severo (vermelho)", fontsize=9)
figura.tight_layout()
figura.savefig("pesos/comparacao.png", dpi=150)
plt.show()

print(f"Modelo: {versao}")
print("Figura salva em pesos/comparacao.png")


## 9. Baixar os pesos

O arquivo tem cerca de 7 MB. Ele guarda apenas os números da rede, a versão e a lista de classes — nunca código.

In [ ]:
%cd /content/SmartAgro/visao
from google.colab import files

files.download("pesos/unet.pt")
files.download("pesos/unet.historico.json")
files.download("pesos/curva-de-treino.png")
files.download("pesos/comparacao.png")


## 10. Usar o modelo no AgroSmart

1. coloque `unet.pt` em `visao/pesos/` no seu computador;
2. no `visao/.env`:

```
VISAO_PESOS=pesos/unet.pt
```

3. rode o serviço:

```bash
.venv/Scripts/python -m visao servico --uma-vez
```

Ele vai anunciar na tela qual modelo carregou, e a versão sai **do arquivo de pesos**, não de uma variável — o que fica registrado com cada análise é o modelo que de fato rodou.

### O que muda no sistema

Com o modelo treinado, a confiança passa a vir **do próprio modelo**, e não de um valor fixo. Análises acima de 70% de confiança seguem direto para o oráculo; abaixo disso continuam indo ao perito (RF17).

Até aqui, com a heurística de cor, **toda** análise ia ao perito, porque a confiança dela é fixa em 45%. Isso era proposital: um número que ninguém conferiu não deve mover dinheiro.

### O que levar para o documento

| O que | De onde |
|---|---|
| Erro do estimador clássico | seção 5 |
| Erro do modelo treinado | seção 7 |
| Curva de treino | `curva-de-treino.png` |
| Comparação visual | `comparacao.png` |
| Citação da base | v2.1, DOI 10.5281/zenodo.22062459, CC BY 4.0 |

### O que dizer sobre os dados, com todas as letras

- O modelo foi treinado em **um único voo, sobre uns 0,3 hectare de milho**, numa só data.
- As máscaras são **rótulos automáticos**, gerados por índices de vegetação calculados com infravermelho. O modelo aprende a prever, pelo visível, um mapa definido pelo infravermelho.
- Aplicar o modelo a **soja**, em São Simão, fotografada com celular, é uma extrapolação. A validação com imagens da lavoura real é trabalho futuro.
- Doença está **fora do escopo**: o modelo não a reconhece.
